# Pengujian Pipeline Lengkap ONNX HAR UAV

Pipeline yang diuji:

`video → YOLOv8s ONNX → ByteTrack → crop → YOLO26s-Pose ONNX → Raw51 → Body110 → CNN-BiLSTM ONNX → wajah/liveness opsional → video dan laporan`

Notebook memakai ONNX CPU untuk menghindari konflik CUDA/ONNX Runtime pada Kaggle. Angka FPS Kaggle bukan prediksi FPS Jetson. Tujuan tahap ini adalah membuktikan integrasi, bentuk data, hasil pose, dan keluaran HAR.

Mode:

- `CORE`: detector, tracker, pose, Body110, dan HAR.
- `FULL`: menambahkan InsightFace dan MiniFASNetV2.

Notebook menggunakan `FULL` sebagai mode bawaan karena dataset `face-assets` sudah tersedia. Gunakan `CORE` hanya jika ingin melakukan pengujian HAR tanpa pengenalan wajah dan liveness.

In [ ]:
# Dependensi. ONNX dijalankan pada CPU agar stabil di semua accelerator Kaggle.
!pip install -q -U ultralytics onnxruntime onnx lap

# InsightFace hanya diperlukan ketika PIPELINE_MODE='FULL'. Kegagalan instalasi
# tidak menghalangi mode CORE.
!pip install -q insightface || true

print("Dependensi selesai.")

In [ ]:
import gc
import json
import math
import platform
import shutil
import subprocess
import time
import warnings
from collections import defaultdict, deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import ultralytics
from IPython.display import Video, display
from ultralytics import YOLO

warnings.filterwarnings("ignore")

print("Python       :", platform.python_version())
print("Torch        :", torch.__version__)
print("GPU tersedia :", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Ultralytics  :", ultralytics.__version__)
print("ORT          :", ort.__version__)
print("ORT providers:", ort.get_available_providers())


## Konfigurasi utama

Konfigurasi memproses seluruh manusia yang lolos confidence dan NMS. Pengujian awal dibatasi 300 frame agar selesai lebih cepat. Setelah pipeline lulus, ubah `MAX_FRAMES = 0` untuk memproses video penuh.

In [ ]:
# ============================================================
# KONFIGURASI
# ============================================================
PIPELINE_MODE = "FULL"       # FULL = HAR + face recognition + liveness

# Kaggle biasanya memasang dataset wafabila/yolo-onnx di folder berikut.
# Pencarian rekursif tetap dipakai sehingga perubahan nama mount tidak masalah.
YOLO_DIR_REQUESTED = Path("/kaggle/input/yolo-onnx")
DETECTOR_FILENAME = "yolov8s_512_fp32.onnx"
POSE_FILENAME = "yolo26s-pose_512_fp32.onnx"

IMGSZ = 512
DETECTOR_CONF = 0.15
POSE_CONF = 0.05
KEYPOINT_CONF = 0.15
MIN_VALID_KEYPOINTS = 5
POSE_INTERVAL = 2
FACE_INTERVAL = 3
FACE_BOX_MAX_AGE = 15       # kotak tetap tampil maksimal 15 frame sejak deteksi terakhir
FACE_BBOX_EMA_ALPHA = 0.65  # bobot kotak lama untuk mengurangi jitter
SEQUENCE_LENGTH = 30
STEP_SIZE = 10
MAX_FRAMES = 300           # 0 = seluruh video
TRACK_STALE_FRAMES = 45
GESTURE_SMOOTHING = 5

FACE_ASSET_ROOT_REQUESTED = Path("/kaggle/input/datasets/wafabila/face-assets/face_assets")
FACE_MODEL_NAME = "buffalo_sc"
FACE_DET_SIZE = (960, 960)
FACE_SIMILARITY_THRESHOLD = 0.39
LIVENESS_THRESHOLD = 0.85
FACE_RESULT_MAX_AGE = 60
FACE_HISTORY_LENGTH = 5
LIVENESS_HISTORY_LENGTH = 30

ONNX_DEVICE = "cpu"
WORK_ROOT = Path("/kaggle/working/FULL_ONNX_HAR_UAV")
OUTPUT_DIR = WORK_ROOT / "outputs"
FRAME_DIR = OUTPUT_DIR / "sample_frames"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

print("Mode:", PIPELINE_MODE)
print("Work:", WORK_ROOT)


In [ ]:
# ============================================================
# PENEMUAN FILE OTOMATIS
# ============================================================
INPUT_ROOT = Path("/kaggle/input")


def find_exact(filename, preferred_text=None, required=True):
    candidates = sorted(INPUT_ROOT.rglob(filename))
    if preferred_text:
        preferred = [path for path in candidates if preferred_text.lower() in str(path).lower()]
        if preferred:
            candidates = preferred
    if candidates:
        return candidates[0]
    if required:
        raise FileNotFoundError(f"{filename} tidak ditemukan di /kaggle/input")
    return None


def find_under(root, filename):
    if root is None or not root.exists():
        return None
    direct = root / filename
    if direct.is_file():
        return direct
    matches = sorted(root.rglob(filename))
    return matches[0] if matches else None


def require_mounted_inputs():
    mounted_files = sorted(path for path in INPUT_ROOT.rglob("*") if path.is_file())
    if not mounted_files:
        raise RuntimeError(
            "/kaggle/input kosong. Klik Stop Session, Add Input untuk yolo-onnx, "
            "model_seed5, dan video_test, lalu Start Session baru. Jalankan sel ini "
            "secara interaktif sebelum Save Version/Run All. Kode tidak dapat memakai "
            "dataset yang belum dipasang oleh Kaggle."
        )
    print(f"Input terpasang: {len(mounted_files)} file")
    for path in mounted_files[:30]:
        print(" -", path)
    return mounted_files


def resolve_yolo_models():
    candidate_roots = [
        YOLO_DIR_REQUESTED,
        Path("/kaggle/input/yolo-onnx"),
        Path("/kaggle/input/yolo_onnx"),
        INPUT_ROOT,
    ]

    for root in candidate_roots:
        detector = find_under(root, DETECTOR_FILENAME)
        pose = find_under(root, POSE_FILENAME)
        if detector is not None and pose is not None:
            print("[YOLO FOUND]", root)
            return detector, pose

    available_onnx = [str(path) for path in INPUT_ROOT.rglob("*.onnx")]
    raise FileNotFoundError(
        f"{DETECTOR_FILENAME} dan/atau {POSE_FILENAME} tidak ditemukan pada input "
        f"yang terpasang. ONNX yang terlihat: {available_onnx}"
    )


mounted_files = require_mounted_inputs()
detector_path, pose_path = resolve_yolo_models()

har_path = find_exact("har_window_30_representative.onnx", "model")
mean_path = find_exact("feature_mean.npy", "model")
std_path = find_exact("feature_std.npy", "model")
metadata_path = find_exact("pipeline_metadata.json", "model")

video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".mpeg", ".mpg", ".m4v"}
video_candidates = sorted(
    path for path in INPUT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in video_extensions
)
preferred_videos = [path for path in video_candidates if "video_test" in str(path).lower()]
video_paths = preferred_videos or video_candidates
if not video_paths:
    raise FileNotFoundError("Tidak ada video pengujian di /kaggle/input")
video_path = video_paths[0]

for path, label in [
    (detector_path, "detector"), (pose_path, "pose"), (har_path, "HAR"),
    (mean_path, "mean"), (std_path, "std"), (metadata_path, "metadata"),
    (video_path, "video"),
]:
    if not path.is_file():
        raise FileNotFoundError(f"{label} tidak ditemukan: {path}")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
class_names = metadata.get("class_names", [])
if not class_names:
    raise KeyError("class_names tidak ditemukan di pipeline_metadata.json")

print("Detector :", detector_path)
print("Pose     :", pose_path)
print("HAR      :", har_path)
print("Mean/std :", mean_path, std_path)
print("Metadata :", metadata_path)
print("Video    :", video_path)
print("Classes  :", class_names)


## Transformasi Body110

Sel berikut disalin dari implementasi yang sama dengan training final. Raw51 `(30,51)` diubah menjadi Body85 dan 25 fitur biomekanik, sehingga masukan HAR menjadi `(1,30,110)`.

In [ ]:
"""Transformasi Raw51 -> Body110 yang konsisten dengan notebook training final."""

from __future__ import annotations

import numpy as np

NUM_KEYPOINTS = 17
RAW_FEATURES_PER_KEYPOINT = 3
MODEL_FEATURES_PER_KEYPOINT = 5
RAW_FEATURE_DIM = 51
BASE_MOTION_FEATURE_DIM = 85
ENGINEERED_FEATURE_DIM = 25
MODEL_FEATURE_DIM = 110

LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_ELBOW, RIGHT_ELBOW = 7, 8
LEFT_WRIST, RIGHT_WRIST = 9, 10
LEFT_HIP, RIGHT_HIP = 11, 12
LEFT_ANKLE, RIGHT_ANKLE = 15, 16

BODY_RELATIVE_CLIP = 4.0
MIN_BODY_SCALE = 0.03

ENGINEERED_FEATURE_NAMES = [
    "left_elbow_angle", "right_elbow_angle",
    "left_elbow_angular_velocity", "right_elbow_angular_velocity",
    "left_wrist_to_shoulder", "right_wrist_to_shoulder",
    "left_arm_extension_velocity", "right_arm_extension_velocity",
    "left_wrist_relative_speed", "right_wrist_relative_speed",
    "left_wrist_relative_horizontal_speed", "right_wrist_relative_horizontal_speed",
    "left_wrist_relative_vertical_speed", "right_wrist_relative_vertical_speed",
    "wrist_distance", "wrist_distance_change", "hand_speed_difference",
    "hand_motion_symmetry", "extension_opposition", "extension_synchrony",
    "hip_center_speed", "shoulder_center_speed",
    "left_ankle_relative_speed", "right_ankle_relative_speed", "stride_width",
]


def _safe_angle(a, b, c, va, vb, vc):
    ba, bc = a - b, c - b
    nba = np.linalg.norm(ba, axis=-1)
    nbc = np.linalg.norm(bc, axis=-1)
    valid = va & vb & vc & (nba > 1e-6) & (nbc > 1e-6)
    cosine = np.sum(ba * bc, axis=-1) / np.maximum(nba * nbc, 1e-6)
    angle = np.arccos(np.clip(cosine, -1.0, 1.0)) / np.pi
    return np.where(valid, angle, 0.0).astype(np.float32)


def _safe_distance(a, b, va, vb):
    return np.where(va & vb, np.linalg.norm(a - b, axis=-1), 0.0).astype(np.float32)


def _safe_speed(velocity, valid):
    return np.where(valid, np.linalg.norm(velocity, axis=-1), 0.0).astype(np.float32)


def _temporal_signed_difference(values, valid):
    output = np.zeros_like(values, dtype=np.float32)
    pair_valid = valid[:, 1:] & valid[:, :-1]
    output[:, 1:] = np.where(pair_valid, values[:, 1:] - values[:, :-1], 0.0)
    return output


def _temporal_absolute_difference(values, valid):
    return np.abs(_temporal_signed_difference(values, valid)).astype(np.float32)


def _motion_symmetry(left_velocity, right_velocity, valid_left, valid_right):
    left_norm = np.linalg.norm(left_velocity, axis=-1)
    right_norm = np.linalg.norm(right_velocity, axis=-1)
    valid = valid_left & valid_right & (left_norm > 1e-6) & (right_norm > 1e-6)
    cosine = np.sum(left_velocity * right_velocity, axis=-1) / np.maximum(
        left_norm * right_norm, 1e-6
    )
    return np.where(valid, (np.clip(cosine, -1.0, 1.0) + 1.0) / 2.0, 0.0).astype(
        np.float32
    )


def _calculate_body_center_and_scale(xy, valid):
    lhv, rhv = valid[..., LEFT_HIP], valid[..., RIGHT_HIP]
    lsv, rsv = valid[..., LEFT_SHOULDER], valid[..., RIGHT_SHOULDER]
    both_hips, both_shoulders = lhv & rhv, lsv & rsv

    hip_center = (xy[..., LEFT_HIP, :] + xy[..., RIGHT_HIP, :]) / 2.0
    shoulder_center = (
        xy[..., LEFT_SHOULDER, :] + xy[..., RIGHT_SHOULDER, :]
    ) / 2.0
    valid_count = np.maximum(valid.sum(axis=-1, keepdims=True), 1)
    valid_mean = (xy * valid[..., None]).sum(axis=-2) / valid_count
    center = np.where(
        both_hips[..., None],
        hip_center,
        np.where(both_shoulders[..., None], shoulder_center, valid_mean),
    )

    torso_length = np.linalg.norm(shoulder_center - hip_center, axis=-1)
    shoulder_width = np.linalg.norm(
        xy[..., LEFT_SHOULDER, :] - xy[..., RIGHT_SHOULDER, :], axis=-1
    )
    hip_width = np.linalg.norm(
        xy[..., LEFT_HIP, :] - xy[..., RIGHT_HIP, :], axis=-1
    )
    scale = np.where(
        both_hips & both_shoulders & (torso_length > MIN_BODY_SCALE),
        torso_length,
        np.where(
            both_shoulders & (shoulder_width > MIN_BODY_SCALE),
            shoulder_width,
            np.where(
                both_hips & (hip_width > MIN_BODY_SCALE), hip_width, MIN_BODY_SCALE
            ),
        ),
    )
    return center.astype(np.float32), np.maximum(scale, MIN_BODY_SCALE).astype(np.float32)


def build_base_motion_features(raw: np.ndarray) -> np.ndarray:
    """(N,T,51) -> (N,T,85): relative_x, relative_y, conf, frame_dx, frame_dy."""
    raw = np.asarray(raw, dtype=np.float32)
    if raw.ndim != 3 or raw.shape[-1] != RAW_FEATURE_DIM:
        raise ValueError(f"Shape Raw51 salah: {raw.shape}")

    n, t, _ = raw.shape
    pose = raw.reshape(n, t, NUM_KEYPOINTS, RAW_FEATURES_PER_KEYPOINT)
    frame_xy, conf = pose[..., :2], pose[..., 2:3]
    valid = conf[..., 0] > 0
    center, scale = _calculate_body_center_and_scale(frame_xy, valid)
    relative_xy = (frame_xy - center[..., None, :]) / scale[..., None, None]
    relative_xy = np.clip(relative_xy, -BODY_RELATIVE_CLIP, BODY_RELATIVE_CLIP)
    relative_xy[~valid] = 0.0

    displacement = np.zeros_like(frame_xy, dtype=np.float32)
    pair_valid = valid[:, 1:] & valid[:, :-1]
    displacement[:, 1:] = (frame_xy[:, 1:] - frame_xy[:, :-1]) * pair_valid[..., None]
    output = np.concatenate([relative_xy, conf, displacement], axis=-1).reshape(
        n, t, BASE_MOTION_FEATURE_DIM
    )
    if not np.isfinite(output).all():
        raise RuntimeError("Body85 mengandung NaN/Inf")
    return output.astype(np.float32)


def append_engineered_features(base_motion: np.ndarray) -> np.ndarray:
    """(N,T,85) -> (N,T,110) dengan 25 fitur biomekanik."""
    base_motion = np.asarray(base_motion, dtype=np.float32)
    if base_motion.ndim != 3 or base_motion.shape[-1] != BASE_MOTION_FEATURE_DIM:
        raise ValueError(f"Shape Body85 salah: {base_motion.shape}")

    pose = base_motion.reshape(
        *base_motion.shape[:2], NUM_KEYPOINTS, MODEL_FEATURES_PER_KEYPOINT
    )
    xy, conf, velocity = pose[..., :2], pose[..., 2], pose[..., 3:5]
    valid = conf > 0
    both_hips = valid[..., LEFT_HIP] & valid[..., RIGHT_HIP]
    both_shoulders = valid[..., LEFT_SHOULDER] & valid[..., RIGHT_SHOULDER]

    left_arm_valid = (
        valid[..., LEFT_SHOULDER] & valid[..., LEFT_ELBOW] & valid[..., LEFT_WRIST]
    )
    right_arm_valid = (
        valid[..., RIGHT_SHOULDER]
        & valid[..., RIGHT_ELBOW]
        & valid[..., RIGHT_WRIST]
    )
    lea = _safe_angle(
        xy[..., LEFT_SHOULDER, :], xy[..., LEFT_ELBOW, :], xy[..., LEFT_WRIST, :],
        valid[..., LEFT_SHOULDER], valid[..., LEFT_ELBOW], valid[..., LEFT_WRIST],
    )
    rea = _safe_angle(
        xy[..., RIGHT_SHOULDER, :], xy[..., RIGHT_ELBOW, :], xy[..., RIGHT_WRIST, :],
        valid[..., RIGHT_SHOULDER], valid[..., RIGHT_ELBOW], valid[..., RIGHT_WRIST],
    )
    leav = _temporal_absolute_difference(lea, left_arm_valid)
    reav = _temporal_absolute_difference(rea, right_arm_valid)

    lwsv = valid[..., LEFT_WRIST] & valid[..., LEFT_SHOULDER]
    rwsv = valid[..., RIGHT_WRIST] & valid[..., RIGHT_SHOULDER]
    lwts = _safe_distance(
        xy[..., LEFT_WRIST, :], xy[..., LEFT_SHOULDER, :],
        valid[..., LEFT_WRIST], valid[..., LEFT_SHOULDER],
    )
    rwts = _safe_distance(
        xy[..., RIGHT_WRIST, :], xy[..., RIGHT_SHOULDER, :],
        valid[..., RIGHT_WRIST], valid[..., RIGHT_SHOULDER],
    )
    laev = _temporal_signed_difference(lwts, lwsv)
    raev = _temporal_signed_difference(rwts, rwsv)

    lwrv = velocity[..., LEFT_WRIST, :] - velocity[..., LEFT_SHOULDER, :]
    rwrv = velocity[..., RIGHT_WRIST, :] - velocity[..., RIGHT_SHOULDER, :]
    lwrs, rwrs = _safe_speed(lwrv, lwsv), _safe_speed(rwrv, rwsv)
    lwrh = np.where(lwsv, np.abs(lwrv[..., 0]), 0.0).astype(np.float32)
    rwrh = np.where(rwsv, np.abs(rwrv[..., 0]), 0.0).astype(np.float32)
    lwrv_speed = np.where(lwsv, np.abs(lwrv[..., 1]), 0.0).astype(np.float32)
    rwrv_speed = np.where(rwsv, np.abs(rwrv[..., 1]), 0.0).astype(np.float32)

    both_wrists = valid[..., LEFT_WRIST] & valid[..., RIGHT_WRIST]
    wrist_distance = _safe_distance(
        xy[..., LEFT_WRIST, :], xy[..., RIGHT_WRIST, :],
        valid[..., LEFT_WRIST], valid[..., RIGHT_WRIST],
    )
    wrist_change = _temporal_absolute_difference(wrist_distance, both_wrists)
    speed_difference = np.where(both_wrists, np.abs(lwrs - rwrs), 0.0).astype(np.float32)
    symmetry = _motion_symmetry(lwrv, rwrv, lwsv, rwsv)
    both_extension = lwsv & rwsv
    opposition = np.where(both_extension, np.abs(laev - raev), 0.0).astype(np.float32)
    synchrony = np.where(both_extension, np.abs(laev + raev), 0.0).astype(np.float32)

    hip_velocity = (velocity[..., LEFT_HIP, :] + velocity[..., RIGHT_HIP, :]) / 2.0
    shoulder_velocity = (
        velocity[..., LEFT_SHOULDER, :] + velocity[..., RIGHT_SHOULDER, :]
    ) / 2.0
    hip_speed = _safe_speed(hip_velocity, both_hips)
    shoulder_speed = _safe_speed(shoulder_velocity, both_shoulders)
    left_ankle_speed = _safe_speed(
        velocity[..., LEFT_ANKLE, :] - hip_velocity,
        valid[..., LEFT_ANKLE] & both_hips,
    )
    right_ankle_speed = _safe_speed(
        velocity[..., RIGHT_ANKLE, :] - hip_velocity,
        valid[..., RIGHT_ANKLE] & both_hips,
    )
    stride_width = _safe_distance(
        xy[..., LEFT_ANKLE, :], xy[..., RIGHT_ANKLE, :],
        valid[..., LEFT_ANKLE], valid[..., RIGHT_ANKLE],
    )

    engineered = np.stack(
        [
            lea, rea, leav, reav, lwts, rwts, laev, raev,
            lwrs, rwrs, lwrh, rwrh, lwrv_speed, rwrv_speed,
            wrist_distance, wrist_change, speed_difference, symmetry,
            opposition, synchrony, hip_speed, shoulder_speed,
            left_ankle_speed, right_ankle_speed, stride_width,
        ],
        axis=-1,
    ).astype(np.float32)
    output = np.concatenate([base_motion, engineered], axis=-1).astype(np.float32)
    if output.shape[-1] != MODEL_FEATURE_DIM or not np.isfinite(output).all():
        raise RuntimeError(f"Body110 tidak valid: {output.shape}")
    return output


def prepare_sequence(raw_sequence, valid_mask, feature_mean, feature_std):
    """Menyiapkan Raw51 (30,51) dan mask (30,) menjadi input ONNX."""
    raw = np.asarray(raw_sequence, dtype=np.float32)
    mask = np.asarray(valid_mask, dtype=bool)
    mean = np.asarray(feature_mean, dtype=np.float32).reshape(-1)
    std = np.asarray(feature_std, dtype=np.float32).reshape(-1)
    if raw.shape != (30, RAW_FEATURE_DIM) or mask.shape != (30,):
        raise ValueError(f"Input sequence salah: raw={raw.shape}, mask={mask.shape}")
    if mean.shape != (MODEL_FEATURE_DIM,) or std.shape != (MODEL_FEATURE_DIM,):
        raise ValueError(f"Scaler harus (110,), diperoleh {mean.shape}/{std.shape}")

    base = build_base_motion_features(raw[None, ...])
    features = append_engineered_features(base)[0]
    base4 = base[0].reshape(30, NUM_KEYPOINTS, MODEL_FEATURES_PER_KEYPOINT)
    keypoint_valid = base4[..., 2] > 0
    transformed = (features - mean[None, :]) / np.maximum(std[None, :], 1e-6)
    transformed_base = transformed[:, :BASE_MOTION_FEATURE_DIM].reshape(
        30, NUM_KEYPOINTS, MODEL_FEATURES_PER_KEYPOINT
    )
    transformed_base[~keypoint_valid] = 0.0
    transformed[:, :BASE_MOTION_FEATURE_DIM] = transformed_base.reshape(
        30, BASE_MOTION_FEATURE_DIM
    )
    transformed[~mask] = 0.0
    if not np.isfinite(transformed).all():
        raise RuntimeError("Input ONNX mengandung NaN/Inf")
    return transformed[None, ...].astype(np.float32), mask[None, ...].astype(np.bool_)



In [ ]:
# ============================================================
# PEMERIKSAAN MODEL DAN SCALER
# ============================================================
feature_mean = np.load(mean_path).astype(np.float32).reshape(-1)
feature_std = np.load(std_path).astype(np.float32).reshape(-1)
assert feature_mean.shape == (110,), feature_mean.shape
assert feature_std.shape == (110,), feature_std.shape
assert np.isfinite(feature_mean).all() and np.isfinite(feature_std).all()

har_session = ort.InferenceSession(str(har_path), providers=["CPUExecutionProvider"])
har_inputs = {item.name: item for item in har_session.get_inputs()}
har_outputs = har_session.get_outputs()
assert "input" in har_inputs and "frame_mask" in har_inputs, list(har_inputs)

detector_model = YOLO(str(detector_path), task="detect")
pose_model = YOLO(str(pose_path), task="pose")

dummy_raw = np.zeros((30, 51), dtype=np.float32)
dummy_mask = np.ones(30, dtype=bool)
dummy_x, dummy_m = prepare_sequence(dummy_raw, dummy_mask, feature_mean, feature_std)
dummy_logits = har_session.run(None, {"input": dummy_x, "frame_mask": dummy_m})[0]

assert dummy_x.shape == (1, 30, 110)
assert dummy_m.shape == (1, 30)
assert dummy_logits.shape == (1, len(class_names)), (dummy_logits.shape, len(class_names))

print("Detector task:", detector_model.task)
print("Pose task    :", pose_model.task)
print("HAR inputs   :", [(item.name, item.shape, item.type) for item in har_session.get_inputs()])
print("HAR outputs  :", [(item.name, item.shape, item.type) for item in har_outputs])
print("Scaler       :", feature_mean.shape, feature_std.shape)
print("Dummy parity : PASS", dummy_x.shape, dummy_logits.shape)


## Modul wajah dan liveness opsional

Pada mode `CORE`, sel ini hanya menyiapkan kelas tanpa memuat model. Pada mode `FULL`, database embedding, InsightFace, dan MiniFASNetV2 dicari dan diaktifkan.

In [ ]:
def normalize_embedding(value):
    value = np.asarray(value, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(value))
    if norm <= 1e-12:
        raise ValueError("Embedding wajah bernilai nol")
    return (value / norm).astype(np.float32)


def resolve_face_assets():
    candidates = [
        FACE_ASSET_ROOT_REQUESTED,
        Path("/kaggle/input/face-assets/face_assets"),
        Path("/kaggle/input/face_assets/face_assets"),
    ]
    for model_path in INPUT_ROOT.rglob("MiniFASNetV2.onnx"):
        candidates.append(model_path.parent.parent)
    for root in candidates:
        embeddings = root / "database" / "embeddings"
        anti_spoof = root / "models" / "MiniFASNetV2.onnx"
        if embeddings.is_dir() and anti_spoof.is_file():
            files = sorted(embeddings.rglob("emb_*.npy"))
            if files:
                return root, embeddings, anti_spoof
    raise FileNotFoundError(
        "Asset wajah tidak lengkap. Dibutuhkan face_assets/database/embeddings/"
        "<identitas>/emb_*.npy dan face_assets/models/MiniFASNetV2.onnx"
    )


def face_status_style(identity, liveness):
    unknown = str(identity).strip().lower() in {"", "unknown", "unrecognized", "none"}
    real = str(liveness).strip().lower() == "real"
    if unknown and not real:
        return "UNKNOWN SPOOF", (0, 0, 255)
    if not unknown and not real:
        return "DETECTED SPOOF", (0, 165, 255)
    if unknown and real:
        return "UNKNOWN REAL", (0, 255, 255)
    return "DETECTED REAL", (0, 255, 0)


class MiniFASNetV2:
    def __init__(self, model_path):
        available = ort.get_available_providers()
        providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in available else []) + ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(model_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        print("[ANTI-SPOOF] Providers:", self.session.get_providers())

    def predict_real_score(self, frame, bbox):
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = [int(v) for v in bbox]
        bw, bh = max(x2 - x1, 1), max(y2 - y1, 1)
        face_ratio = bh / max(h, 1)
        base_scale = 3.5 if 0.15 < face_ratio < 0.35 else 2.7
        scale = min((h - 1) / bh, (w - 1) / bw, base_scale)
        cx, cy = x1 + bw / 2, y1 + bh / 2
        nw, nh = bw * scale, bh * scale
        ax1, ay1 = max(0, int(cx - nw / 2)), max(0, int(cy - nh / 2))
        ax2, ay2 = min(w - 1, int(cx + nw / 2)), min(h - 1, int(cy + nh / 2))
        crop = frame[ay1:ay2 + 1, ax1:ax2 + 1]
        if crop.size == 0:
            return 0.0
        tensor = cv2.resize(crop, (80, 80)).astype(np.float32)
        tensor = np.transpose(tensor, (2, 0, 1))[None]
        logits = self.session.run(None, {self.input_name: tensor})[0][0]
        shifted = logits - np.max(logits)
        probabilities = np.exp(shifted) / np.maximum(np.exp(shifted).sum(), 1e-8)
        return float(probabilities[1])


class FaceSystem:
    def __init__(self, asset_root, embedding_root, anti_spoof_path):
        from insightface.app import FaceAnalysis

        self.database = defaultdict(list)
        for person_dir in sorted(embedding_root.iterdir()):
            if not person_dir.is_dir():
                continue
            for file in sorted(person_dir.glob("emb_*.npy")):
                embedding = np.load(file, allow_pickle=False).astype(np.float32)
                if embedding.shape != (512,):
                    raise ValueError(f"Dimensi embedding salah: {file} {embedding.shape}")
                self.database[person_dir.name].append(normalize_embedding(embedding))
        total = sum(len(items) for items in self.database.values())
        if total == 0:
            raise RuntimeError(f"Database embedding kosong: {embedding_root}")
        for name, items in self.database.items():
            print(f"[FACE DB] {name}: {len(items)} embedding")
        print(f"[FACE DB] {len(self.database)} identitas, {total} embedding")

        available = ort.get_available_providers()
        providers = (["CUDAExecutionProvider"] if "CUDAExecutionProvider" in available else []) + ["CPUExecutionProvider"]
        bundled_buffalo = asset_root / "models" / FACE_MODEL_NAME
        insight_root = asset_root if bundled_buffalo.is_dir() else Path("/kaggle/working/insightface")
        insight_root.mkdir(parents=True, exist_ok=True)
        self.app = FaceAnalysis(
            name=FACE_MODEL_NAME, root=str(insight_root),
            allowed_modules=["detection", "recognition"], providers=providers,
        )
        ctx_id = 0 if "CUDAExecutionProvider" in providers else -1
        self.app.prepare(ctx_id=ctx_id, det_size=FACE_DET_SIZE)
        self.anti_spoof = MiniFASNetV2(anti_spoof_path)
        self.identity_history = defaultdict(lambda: deque(maxlen=FACE_HISTORY_LENGTH))
        self.liveness_history = defaultdict(lambda: deque(maxlen=LIVENESS_HISTORY_LENGTH))
        print("[FACE] Providers:", providers)

    def recognize(self, embedding):
        query = normalize_embedding(embedding)
        best_name, best_score = "Unknown", -1.0
        for name, values in self.database.items():
            for stored in values:
                score = float(np.dot(query, stored))
                if score > best_score:
                    best_name, best_score = name, score
        if best_score < FACE_SIMILARITY_THRESHOLD:
            best_name = "Unknown"
        return best_name, best_score

    @staticmethod
    def match_track(face_box, track_boxes, track_scores):
        fx1, fy1, fx2, fy2 = map(float, face_box)
        fcx, fcy = (fx1 + fx2) / 2, (fy1 + fy2) / 2
        candidates = []
        for track_id, box in track_boxes.items():
            x1, y1, x2, y2 = map(float, box)
            bw, bh = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
            inside = (x1 - 0.20 * bw <= fcx <= x2 + 0.20 * bw and y1 - 0.30 * bh <= fcy <= y2 + 0.10 * bh)
            horizontal = abs(fcx - (x1 + x2) / 2) / bw
            vertical = abs(fcy - (y1 + 0.12 * bh)) / bh
            outside_penalty = 0.0 if inside else 2.0
            detector_penalty = (1.0 - float(track_scores.get(track_id, 0.0))) * 0.10
            candidates.append((horizontal + vertical + outside_penalty + detector_penalty, int(track_id)))
        if not candidates:
            return None
        best_score, best_track = min(candidates)
        return best_track if best_score <= 2.50 else None

    def process(self, frame, track_boxes, track_scores):
        output = {}
        for face in self.app.get(frame):
            track_id = self.match_track(face.bbox, track_boxes, track_scores)
            if track_id is None:
                continue
            identity, similarity = self.recognize(face.embedding)
            raw_real_score = self.anti_spoof.predict_real_score(frame, face.bbox)
            self.identity_history[track_id].append({"identity": identity, "similarity": similarity})
            self.liveness_history[track_id].append(raw_real_score)
            grouped = defaultdict(list)
            for item in self.identity_history[track_id]:
                grouped[item["identity"]].append(item["similarity"])
            selected = max(grouped, key=lambda name: (len(grouped[name]), float(np.mean(grouped[name]))))
            smooth_similarity = float(np.mean(grouped[selected]))
            smooth_real_score = float(np.mean(self.liveness_history[track_id]))
            liveness = "Real" if smooth_real_score > LIVENESS_THRESHOLD else "Spoof"
            status, color = face_status_style(selected, liveness)
            output[track_id] = {
                "track_id": track_id, "bbox": np.asarray(face.bbox, dtype=np.int32),
                "identity": selected, "similarity": smooth_similarity,
                "liveness": liveness, "liveness_score": smooth_real_score,
                "raw_liveness_score": float(raw_real_score), "status": status, "color": color,
            }
        return output

    def forget(self, track_id):
        self.identity_history.pop(track_id, None)
        self.liveness_history.pop(track_id, None)


def project_cached_face_bbox(result, current_body_box, frame_shape):
    """Proyeksikan kotak wajah terakhir mengikuti perubahan bbox tubuh dari ByteTrack."""
    face_box = np.asarray(result["bbox"], dtype=np.float32).copy()
    source_body = result.get("body_bbox")
    if source_body is not None and current_body_box is not None:
        sx1, sy1, sx2, sy2 = np.asarray(source_body, dtype=np.float32)
        cx1, cy1, cx2, cy2 = np.asarray(current_body_box, dtype=np.float32)
        source_w, source_h = max(sx2 - sx1, 1.0), max(sy2 - sy1, 1.0)
        current_w, current_h = max(cx2 - cx1, 1.0), max(cy2 - cy1, 1.0)
        face_box[[0, 2]] = cx1 + ((face_box[[0, 2]] - sx1) / source_w) * current_w
        face_box[[1, 3]] = cy1 + ((face_box[[1, 3]] - sy1) / source_h) * current_h
    height, width = frame_shape[:2]
    face_box[[0, 2]] = np.clip(face_box[[0, 2]], 0, width - 1)
    face_box[[1, 3]] = np.clip(face_box[[1, 3]], 0, height - 1)
    return np.rint(face_box).astype(np.int32)


def draw_face_box(frame, result):
    x1, y1, x2, y2 = [int(v) for v in result["bbox"]]
    color = tuple(result["color"])
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
    text = f"{result['status']} | T{result['track_id']} | {result['identity']} {result['similarity']:.2f} | {result['liveness']} {result['liveness_score']:.2f}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.48, 2)
    ty1 = max(0, y1 - th - 12)
    cv2.rectangle(frame, (x1, ty1), (min(frame.shape[1] - 1, x1 + tw + 10), y1), color, -1)
    text_color = (0, 0, 0) if color != (0, 0, 255) else (255, 255, 255)
    cv2.putText(frame, text, (x1 + 5, max(th + 2, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.48, text_color, 2, cv2.LINE_AA)


face_system = None
FACE_ASSET_ROOT = FACE_EMBEDDINGS_DIR = MINIFASNET_PATH = None
if PIPELINE_MODE.upper() == "FULL":
    FACE_ASSET_ROOT, FACE_EMBEDDINGS_DIR, MINIFASNET_PATH = resolve_face_assets()
    face_system = FaceSystem(FACE_ASSET_ROOT, FACE_EMBEDDINGS_DIR, MINIFASNET_PATH)
    print("Face FULL aktif")
    print("Asset root :", FACE_ASSET_ROOT)
    print("Embeddings :", FACE_EMBEDDINGS_DIR)
    print("MiniFASNet :", MINIFASNET_PATH)
else:
    print("Mode CORE: face dan liveness dilewati.")


## Fungsi pipeline video

In [ ]:
SKELETON_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
    (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
    (13, 15), (12, 14), (14, 16),
]


def padded_box(box, width, height, pad_x=0.25, pad_y=0.35):
    x1, y1, x2, y2 = map(float, box)
    bw, bh = max(x2 - x1, 1.0), max(y2 - y1, 1.0)
    return (
        max(0, int(x1 - bw * pad_x)), max(0, int(y1 - bh * pad_y)),
        min(width, int(x2 + bw * pad_x)), min(height, int(y2 + bh * pad_y)),
    )


def pose_to_raw51(frame, box):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = padded_box(box, width, height)
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros(51, np.float32), False

    result = pose_model.predict(
        crop, imgsz=IMGSZ, conf=POSE_CONF, iou=0.50,
        classes=[0], device=ONNX_DEVICE, verbose=False,
    )[0]
    if result.boxes is None or result.keypoints is None or len(result.boxes) == 0:
        return np.zeros(51, np.float32), False

    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)
    scores = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    keypoints = result.keypoints.data.detach().cpu().numpy().astype(np.float32)
    assert keypoints.shape[1] == 17 and keypoints.shape[2] >= 3, keypoints.shape
    areas = np.maximum(boxes[:, 2] - boxes[:, 0], 0) * np.maximum(boxes[:, 3] - boxes[:, 1], 0)
    best = int(np.argmax(areas * np.maximum(scores, 1e-6)))

    pose = keypoints[best, :17, :3].copy()
    pose[:, 0] = (pose[:, 0] + x1) / max(width, 1)
    pose[:, 1] = (pose[:, 1] + y1) / max(height, 1)
    pose[:, :2] = np.clip(pose[:, :2], 0.0, 1.0)
    valid = pose[:, 2] >= KEYPOINT_CONF
    pose[~valid] = 0.0
    frame_valid = int(valid.sum()) >= MIN_VALID_KEYPOINTS
    if not frame_valid:
        pose[:] = 0.0
    return pose.reshape(51).astype(np.float32), frame_valid


def softmax(logits):
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.maximum(exp.sum(axis=-1, keepdims=True), 1e-8)


def infer_har(raw_buffer, mask_buffer):
    model_input, mask = prepare_sequence(
        np.asarray(raw_buffer), np.asarray(mask_buffer), feature_mean, feature_std
    )
    logits = har_session.run(None, {"input": model_input, "frame_mask": mask})[0]
    probability = softmax(logits)[0]
    assert probability.shape == (len(class_names),)
    return probability.astype(np.float32)


def draw_pose(frame, raw51):
    h, w = frame.shape[:2]
    pose = np.asarray(raw51).reshape(17, 3)
    valid = pose[:, 2] > 0
    points = np.stack([pose[:, 0] * w, pose[:, 1] * h], axis=-1).astype(int)
    for a, b in SKELETON_EDGES:
        if valid[a] and valid[b]:
            cv2.line(frame, tuple(points[a]), tuple(points[b]), (0, 255, 255), 2)
    for index, point in enumerate(points):
        if valid[index]:
            cv2.circle(frame, tuple(point), 3, (0, 80, 255), -1)


def draw_track(frame, box, lines, ready=False):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    color = (0, 220, 0) if ready else (0, 165, 255)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness, spacing = 0.56, 2, 24
    text_width = max(cv2.getTextSize(line, font, scale, thickness)[0][0] for line in lines)
    panel_width = min(text_width + 18, width - max(x1, 0))
    panel_height = spacing * len(lines) + 10
    panel_x1 = max(0, min(x1, width - panel_width))
    panel_y1 = y1 - panel_height if y1 >= panel_height else min(y1 + 3, height - panel_height)
    panel_x2 = min(width - 1, panel_x1 + panel_width)
    panel_y2 = min(height - 1, panel_y1 + panel_height)

    overlay = frame.copy()
    cv2.rectangle(overlay, (panel_x1, panel_y1), (panel_x2, panel_y2), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.78, frame, 0.22, 0, frame)
    cv2.rectangle(frame, (panel_x1, panel_y1), (panel_x2, panel_y2), color, 2)

    text_y = panel_y1 + 21
    for index, line in enumerate(lines):
        text_color = color if index == 0 else (255, 255, 255)
        cv2.putText(frame, line, (panel_x1 + 8, text_y), font, scale,
                    text_color, thickness, cv2.LINE_AA)
        text_y += spacing


## Jalankan pipeline

Cell ini menghasilkan video beranotasi, prediksi HAR per window, audit frame, serta waktu setiap modul.

In [ ]:
capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise RuntimeError(f"Video gagal dibuka: {video_path}")

source_fps = float(capture.get(cv2.CAP_PROP_FPS))
width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_source_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
if not np.isfinite(source_fps) or source_fps <= 0:
    source_fps = 30.0

temporary_video = OUTPUT_DIR / f"{video_path.stem}_{PIPELINE_MODE.lower()}_onnx_mp4v.mp4"
output_video = OUTPUT_DIR / f"{video_path.stem}_{PIPELINE_MODE.lower()}_onnx_h264.mp4"
writer = cv2.VideoWriter(
    str(temporary_video), cv2.VideoWriter_fourcc(*"mp4v"),
    min(source_fps, 30.0), (width, height),
)
if not writer.isOpened():
    capture.release()
    raise RuntimeError("VideoWriter gagal")

raw_buffers = defaultdict(lambda: deque(maxlen=SEQUENCE_LENGTH))
mask_buffers = defaultdict(lambda: deque(maxlen=SEQUENCE_LENGTH))
probability_history = defaultdict(lambda: deque(maxlen=GESTURE_SMOOTHING))
samples_seen = defaultdict(int)
last_seen, last_pose, last_prediction, last_face, last_face_frame = {}, {}, {}, {}, {}

timings = defaultdict(list)
prediction_rows = []
face_rows = []
frame_rows = []
frame_index = 0
fps_history = deque(maxlen=30)
started_all = time.perf_counter()

while True:
    frame_started = time.perf_counter()
    ok, frame = capture.read()
    if not ok:
        break
    if MAX_FRAMES > 0 and frame_index >= MAX_FRAMES:
        break

    tick = time.perf_counter()
    tracked = detector_model.track(
        frame, persist=True, tracker="bytetrack.yaml", classes=[0],
        conf=DETECTOR_CONF, iou=0.50, imgsz=IMGSZ,
        device=ONNX_DEVICE, verbose=False,
    )[0]
    timings["detector_bytetrack_ms"].append((time.perf_counter() - tick) * 1000)

    detections = []
    current_boxes = {}
    current_scores = {}
    if tracked.boxes is not None and len(tracked.boxes) > 0:
        boxes = tracked.boxes.xyxy.detach().cpu().numpy()
        confs = tracked.boxes.conf.detach().cpu().numpy()
        ids_tensor = tracked.boxes.id
        ids = ids_tensor.detach().cpu().numpy().astype(int) if ids_tensor is not None else np.arange(len(boxes))
        detections = list(zip(boxes, confs, ids))
        current_boxes = {int(track_id): box for box, _, track_id in detections}
        current_scores = {int(track_id): float(score) for _, score, track_id in detections}

    if face_system is not None and frame_index % FACE_INTERVAL == 0:
        tick = time.perf_counter()
        face_updates = face_system.process(frame, current_boxes, current_scores)
        for face_track_id, face_result in face_updates.items():
            current_body_box = current_boxes.get(face_track_id)
            previous_face = last_face.get(face_track_id)
            if previous_face is not None and current_body_box is not None:
                previous_projected = project_cached_face_bbox(previous_face, current_body_box, frame.shape)
                detected_bbox = np.asarray(face_result["bbox"], dtype=np.float32)
                face_result["bbox"] = np.rint(
                    FACE_BBOX_EMA_ALPHA * previous_projected
                    + (1.0 - FACE_BBOX_EMA_ALPHA) * detected_bbox
                ).astype(np.int32)
            if current_body_box is not None:
                face_result["body_bbox"] = np.asarray(current_body_box, dtype=np.float32).copy()
            last_face[face_track_id] = face_result
            last_face_frame[face_track_id] = frame_index
            face_rows.append({
                "frame_index": frame_index,
                "time_seconds": frame_index / source_fps,
                "track_id": int(face_track_id),
                "identity": face_result["identity"],
                "similarity": float(face_result["similarity"]),
                "liveness": face_result["liveness"],
                "liveness_score": float(face_result["liveness_score"]),
                "face_status": face_result["status"],
            })
        timings["face_liveness_ms"].append((time.perf_counter() - tick) * 1000)

    for box, detector_score, track_id in detections:
        track_id = int(track_id)
        run_pose = frame_index % POSE_INTERVAL == 0 or track_id not in last_pose
        if run_pose:
            tick = time.perf_counter()
            raw51, frame_valid = pose_to_raw51(frame, box)
            timings["pose_ms"].append((time.perf_counter() - tick) * 1000)
            last_pose[track_id] = raw51
        else:
            raw51 = last_pose[track_id].copy()
            frame_valid = int(np.count_nonzero(raw51.reshape(17, 3)[:, 2])) >= MIN_VALID_KEYPOINTS

        raw_buffers[track_id].append(raw51)
        mask_buffers[track_id].append(frame_valid)
        samples_seen[track_id] += 1
        last_seen[track_id] = frame_index

        if len(raw_buffers[track_id]) == SEQUENCE_LENGTH and samples_seen[track_id] % STEP_SIZE == 0:
            tick = time.perf_counter()
            probabilities = infer_har(raw_buffers[track_id], mask_buffers[track_id])
            timings["body110_har_ms"].append((time.perf_counter() - tick) * 1000)
            probability_history[track_id].append(probabilities)
            smooth = np.mean(probability_history[track_id], axis=0)
            class_id = int(np.argmax(smooth))
            last_prediction[track_id] = (class_names[class_id], float(smooth[class_id]))
            face_for_prediction = last_face.get(track_id)
            face_fresh_for_prediction = (
                face_for_prediction is not None
                and frame_index - last_face_frame.get(track_id, -10000) <= FACE_RESULT_MAX_AGE
            )
            prediction_rows.append({
                "frame_index": frame_index,
                "time_seconds": frame_index / source_fps,
                "track_id": track_id,
                "activity": class_names[class_id],
                "confidence": float(smooth[class_id]),
                "confidence_percent": float(smooth[class_id] * 100.0),
                "valid_pose_frames": int(sum(mask_buffers[track_id])),
                "identity": face_for_prediction["identity"] if face_fresh_for_prediction else "Unknown",
                "face_similarity": float(face_for_prediction["similarity"]) if face_fresh_for_prediction else np.nan,
                "liveness": face_for_prediction["liveness"] if face_fresh_for_prediction else "Not detected",
                "liveness_score": float(face_for_prediction["liveness_score"]) if face_fresh_for_prediction else np.nan,
                "face_status": face_for_prediction["status"] if face_fresh_for_prediction else "NOT DETECTED",
            })

        draw_pose(frame, raw51)
        prediction_ready = track_id in last_prediction
        activity, activity_score = last_prediction.get(track_id, ("MENUNGGU BUFFER", 0.0))
        if prediction_ready:
            activity_line = f"ID {track_id} | {activity.upper()} | {activity_score * 100.0:.2f}%"
        else:
            activity_line = f"ID {track_id} | HAR: MENUNGGU {len(raw_buffers[track_id])}/{SEQUENCE_LENGTH}"
        lines = [
            activity_line,
            f"Deteksi {float(detector_score) * 100.0:.1f}% | Pose valid {int(sum(mask_buffers[track_id]))}/{len(mask_buffers[track_id])}",
        ]
        face_is_fresh = (
            track_id in last_face
            and frame_index - last_face_frame.get(track_id, -10000) <= FACE_RESULT_MAX_AGE
        )
        if face_is_fresh:
            face = last_face[track_id]
            lines.append(
                f"Face: {face['status']} | {face['identity']} {face['similarity']:.2f} | "
                f"{face['liveness']} {face['liveness_score']:.2f}"
            )
        elif face_system is not None:
            lines.append("Face: not detected")
        draw_track(frame, box, lines, ready=prediction_ready)
        face_box_is_fresh = (
            face_is_fresh
            and frame_index - last_face_frame.get(track_id, -10000) <= FACE_BOX_MAX_AGE
        )
        if face_box_is_fresh:
            displayed_face = dict(last_face[track_id])
            displayed_face["bbox"] = project_cached_face_bbox(
                last_face[track_id], box, frame.shape
            )
            draw_face_box(frame, displayed_face)

    stale = [track_id for track_id, seen in last_seen.items() if frame_index - seen > TRACK_STALE_FRAMES]
    for track_id in stale:
        for store in [raw_buffers, mask_buffers, probability_history, samples_seen,
                      last_seen, last_pose, last_prediction, last_face, last_face_frame]:
            store.pop(track_id, None)
        if face_system is not None:
            face_system.forget(track_id)

    frame_ms = (time.perf_counter() - frame_started) * 1000
    timings["total_frame_ms"].append(frame_ms)
    fps_now = 1000.0 / max(frame_ms, 1e-6)
    fps_history.append(fps_now)
    fps_smooth = float(np.mean(fps_history))
    fresh_face_count = sum(
        frame_index - last_face_frame.get(track_id, -10000) <= FACE_BOX_MAX_AGE
        for track_id in current_boxes
    ) if face_system is not None else 0
    header = (
        f"ONLINE FRAME-BY-FRAME | Orang: {len(detections)} | Wajah: {fresh_face_count} | "
        f"FPS: {fps_smooth:.2f} | Frame: {frame_index}"
    )
    header_size = cv2.getTextSize(header, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)[0]
    cv2.rectangle(frame, (5, 5), (min(width - 1, header_size[0] + 23), 39), (15, 15, 15), -1)
    cv2.putText(
        frame, header,
        (12, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2,
        cv2.LINE_AA,
    )
    writer.write(frame)

    frame_rows.append({
        "frame_index": frame_index,
        "time_seconds": frame_index / source_fps,
        "people": len(detections),
        "frame_ms": frame_ms,
        "instant_fps": fps_now,
        "smoothed_fps": fps_smooth,
    })

    if frame_index in {0, 29, 59, 119, 299}:
        cv2.imwrite(str(FRAME_DIR / f"frame_{frame_index:06d}.jpg"), frame)

    frame_index += 1

capture.release()
writer.release()
wall_seconds = time.perf_counter() - started_all

# H.264 lebih mudah diputar pada browser/Kaggle dibanding MP4V.
ffmpeg_command = [
    "ffmpeg", "-y", "-loglevel", "error", "-i", str(temporary_video),
    "-c:v", "libx264", "-preset", "fast", "-crf", "23",
    "-pix_fmt", "yuv420p", str(output_video),
]
ffmpeg_result = subprocess.run(ffmpeg_command, check=False)
if ffmpeg_result.returncode != 0 or not output_video.is_file():
    print("Peringatan: konversi H.264 gagal; memakai MP4V sebagai fallback.")
    output_video = temporary_video

print("Selesai memproses", frame_index, "frame")
print("Video output:", output_video)
print("Wall time   :", wall_seconds)
print("Throughput  :", frame_index / max(wall_seconds, 1e-6), "FPS")


## Laporan, visualisasi, dan ZIP

In [ ]:
prediction_df = pd.DataFrame(prediction_rows)
frame_df = pd.DataFrame(frame_rows)
face_df = pd.DataFrame(face_rows)
prediction_csv = OUTPUT_DIR / "har_predictions.csv"
frame_csv = OUTPUT_DIR / "frame_audit.csv"
face_csv = OUTPUT_DIR / "face_predictions.csv"
prediction_df.to_csv(prediction_csv, index=False)
frame_df.to_csv(frame_csv, index=False)
face_df.to_csv(face_csv, index=False)


def summarize(values):
    if not values:
        return {"count": 0, "mean_ms": None, "p95_ms": None, "max_ms": None}
    array = np.asarray(values, dtype=np.float64)
    return {
        "count": int(array.size),
        "mean_ms": float(array.mean()),
        "p95_ms": float(np.percentile(array, 95)),
        "max_ms": float(array.max()),
    }


report = {
    "pipeline_mode": PIPELINE_MODE,
    "video": str(video_path),
    "frames_processed": frame_index,
    "source_fps": source_fps,
    "source_total_frames": total_source_frames,
    "wall_seconds": wall_seconds,
    "throughput_fps": frame_index / max(wall_seconds, 1e-6),
    "configuration": {
        "imgsz": IMGSZ,
        "people_limit": "none (Ultralytics default max_det)",
        "pose_interval": POSE_INTERVAL,
        "face_interval": FACE_INTERVAL,
        "face_box_max_age": FACE_BOX_MAX_AGE,
        "face_bbox_ema_alpha": FACE_BBOX_EMA_ALPHA,
        "face_result_max_age": FACE_RESULT_MAX_AGE,
        "face_identity_history": FACE_HISTORY_LENGTH,
        "liveness_history": LIVENESS_HISTORY_LENGTH,
        "face_similarity_threshold": FACE_SIMILARITY_THRESHOLD,
        "liveness_threshold": LIVENESS_THRESHOLD,
        "window": SEQUENCE_LENGTH,
        "step": STEP_SIZE,
        "onnx_device": ONNX_DEVICE,
    },
    "models": {
        "detector": str(detector_path),
        "pose": str(pose_path),
        "har": str(har_path),
        "face_model": FACE_MODEL_NAME if face_system is not None else None,
        "face_embeddings": str(FACE_EMBEDDINGS_DIR) if FACE_EMBEDDINGS_DIR else None,
        "anti_spoof": str(MINIFASNET_PATH) if MINIFASNET_PATH else None,
    },
    "prediction_count": len(prediction_rows),
    "face_prediction_count": len(face_rows),
    "module_timing": {name: summarize(values) for name, values in timings.items()},
    "warning": "Throughput Kaggle ONNX CPU bukan FPS TensorRT Jetson.",
}

report_path = OUTPUT_DIR / "pipeline_benchmark.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(json.dumps(report, indent=2))
display(prediction_df.head(20))
if not face_df.empty:
    display(face_df.head(20))
display(frame_df.describe(include="all"))

sample_images = sorted(FRAME_DIR.glob("*.jpg"))
if sample_images:
    columns = 2
    rows = math.ceil(len(sample_images) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 6 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, image_path in zip(axes, sample_images):
        image = cv2.imread(str(image_path))
        axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axis.set_title(image_path.name)
        axis.axis("off")
    for axis in axes[len(sample_images):]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

zip_path = Path(shutil.make_archive(
    "/kaggle/working/FULL_ONNX_HAR_UAV_RESULT", "zip", root_dir=OUTPUT_DIR
))

print("=" * 80)
print("OUTPUT VIDEO:", output_video)
print("PREDICTIONS :", prediction_csv)
print("FACE CSV    :", face_csv)
print("FRAME AUDIT :", frame_csv)
print("BENCHMARK   :", report_path)
print("ZIP         :", zip_path)
print("=" * 80)
print("Klik Save Version untuk mempertahankan output Kaggle.")


In [ ]:
# Preview video di notebook
if output_video.is_file():
    display(Video(str(output_video), embed=True, width=900))


## Interpretasi hasil

Pipeline dinyatakan terintegrasi jika:

1. video output terbentuk dan dapat diputar;
2. bounding box memiliki Track ID stabil;
3. skeleton memiliki 17 keypoint;
4. buffer pose mencapai 30 frame;
5. `har_predictions.csv` berisi kelas aktivitas;
6. pada mode `FULL`, kotak wajah menampilkan identitas, similarity, serta status `Real`/`Spoof`;
7. `face_predictions.csv` berisi hasil wajah per Track ID;
8. `pipeline_benchmark.json` tidak mengandung error atau NaN.

Mode bawaan notebook adalah `FULL`. Ubah ke `CORE` hanya untuk mengukur pipeline HAR tanpa tambahan beban pengenalan wajah dan liveness. Jangan membandingkan FPS ONNX CPU Kaggle dengan TensorRT Jetson sebagai angka yang setara.